# Paper figures and charts

Run all cells in order, or run `python -m biomecg.reproduce` from the checkout. This notebook redraws the paper figures from the supplied frozen aggregate results. It needs no clinical data or GPU. Training and recomputation are documented separately in the README.

Only MIMIC and CODE-15 experiments are included. UMAP uses supplied coordinates and arbitrary display labels, with no clinical records or cohort identifiers. Missing experimental combinations remain explicitly marked in tables and the seed-count figure.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, ticker
from matplotlib.lines import Line2D
from IPython.display import display
from biomecg.paths import ROOT
from biomecg.paper import snapshot
import json
OUT = ROOT / 'outputs/paper'
import os
ARTIFACTS = Path(os.environ.get('BIOMECG_ARTIFACTS', str(ROOT / 'artifacts/paper')))
for folder in ('figures', 'tables'):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
PAPER_WIDTH_IN = 6.5
MODEL_STYLE = {'scratch': dict(label='biomECG scratch', color='#356AA0'), 'ecgfounder': dict(label='ECGFounder', color='#C87535'), 'mimic': dict(label='biomECG-1k (MIMIC)', color='#356AA0'), 'code15': dict(label='biomECG-1k (CODE-15)', color='#356AA0')}
REGIME_STYLE = {'frozen': dict(linestyle='-', marker='o'), 'finetuned': dict(linestyle='-', marker='o'), 'scratch': dict(linestyle='-', marker='s')}
FAMILY_COLORS = {'resnet': {'late': '#356AA0', 'early': '#95BBDD'}, 'ecgfounder': {'late': '#C87535', 'early': '#EAB78F'}}
SOURCE_LINESTYLES = {'mimic': '-', 'code15': '--'}
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans'], 'mathtext.fontset': 'dejavusans', 'font.size': 9, 'axes.labelsize': 10, 'axes.labelweight': 'bold', 'axes.labelpad': 1.5, 'axes.titlesize': 11, 'figure.titlesize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 8.5, 'legend.title_fontsize': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'axes.axisbelow': True, 'axes.facecolor': 'white', 'figure.facecolor': 'white', 'axes.grid': True, 'grid.color': '#D9DEE5', 'grid.alpha': 0.6, 'grid.linewidth': 0.6, 'lines.linewidth': 1.1, 'lines.markersize': 4.5, 'errorbar.capsize': 2.5, 'legend.frameon': True, 'legend.framealpha': 0.9, 'legend.edgecolor': '#D8D5DF', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.dpi': 110, 'savefig.dpi': 300})
print('Exports: outputs/paper')
EXCLUDED_DOWNSTREAM_COHORTS = {'100000'}


In [ ]:
def paper_subplots(nrows=1, ncols=1, *, height=None, width=None, **kwargs):
    """Create figures at their intended printed size; tune height for dense panels."""
    width = PAPER_WIDTH_IN if width is None else width
    height = max(2.8, 2.5 * nrows) if height is None else height
    kwargs.setdefault('layout', 'constrained')
    return plt.subplots(nrows, ncols, figsize=(width, height), **kwargs)

def curve_style(source, *, fusion=None, regime=None):
    """Family = hue; fusion = shade; regime = marker fill/shape."""
    fusion = (fusion or 'late').lower()
    regime = regime or ('scratch' if source == 'scratch' else 'frozen')
    family = 'ecgfounder' if source == 'ecgfounder' else 'resnet'
    color = FAMILY_COLORS[family][fusion]
    return dict(color=color, **REGIME_STYLE[regime], linewidth=1.1, markersize=3.2, markeredgewidth=0.7, alpha=0.85, markerfacecolor=color if regime == 'finetuned' else 'white', markeredgecolor=color)

def training_size_axis(ax, sizes, *, full_n=None, log=True):
    """Return true numeric x positions; Full must be explicitly resolved."""
    positions, labels = ([], [])
    for size in sizes:
        if str(size) in EXCLUDED_DOWNSTREAM_COHORTS:
            raise ValueError('Downstream cohort 100000 is excluded throughout this notebook.')
        if str(size).lower() == 'full':
            if full_n is None:
                raise ValueError('Full requires the actual number of training patients.')
            value, label = (float(full_n), f'Full\n({int(full_n):,})')
        else:
            value, label = (float(size), f'{int(size):,}')
        if not np.isfinite(value) or value <= 0 or value != int(value):
            raise ValueError('Training sizes must be positive integer patient counts.')
        positions.append(value)
        labels.append(label)
    if len(set(positions)) != len(positions):
        raise ValueError('Duplicate training sizes; merge equivalent Full/numeric entries.')
    ax.set_xscale('log' if log else 'linear')
    if log:
        ax.xaxis.set_major_locator(ticker.LogLocator(base=10, subs=(1.0,), numticks=12))
        ax.xaxis.set_major_formatter(ticker.LogFormatterMathtext(base=10, labelOnlyBase=True))
        ax.xaxis.set_minor_locator(ticker.LogLocator(base=10, subs=np.arange(2, 10)))
        ax.xaxis.set_minor_formatter(ticker.NullFormatter())
        ax.grid(which='minor', axis='x', alpha=0.15, linewidth=0.4)
    else:
        ax.set_xticks(positions, labels)
    ax.grid(which='major', alpha=0.65, linewidth=0.6)
    if positions:
        lower, upper = (min(positions), max(positions))
        if log:
            ax.set_xlim(lower / 1.12, upper * 1.12)
        else:
            pad = max((upper - lower) * 0.05, 1)
            ax.set_xlim(lower - pad, upper + pad)
    ax.set_xlabel('Downstream training size (patients)')
    return np.asarray(positions)

def save_paper_figure(fig, name, *, show=True):
    """Export without tight cropping to preserve the configured physical width."""
    if not re.fullmatch('[A-Za-z0-9_-]+', name):
        raise ValueError('Use a simple export name, without a directory or extension.')
    for extension in ('pdf', 'png'):
        fig.savefig(OUT / 'figures' / f'{name}.{extension}', dpi=300, facecolor='white')
    if show:
        plt.show()
    plt.close(fig)


In [ ]:
def latex_escape(value):
    replacements = {'\\': '\\textbackslash{}', '&': '\\&', '%': '\\%', '$': '\\$', '#': '\\#', '_': '\\_', '{': '\\{', '}': '\\}', '~': '\\textasciitilde{}', '^': '\\textasciicircum{}'}
    return ''.join((replacements.get(c, c) for c in str(value)))

def format_estimate(mean, sd=None, n=None, *, decimals=3):
    """Plain-text mean ± SD for display and export; never invent a missing SD."""
    if mean is None or pd.isna(mean):
        return '—'
    text = f'{mean:.{decimals}f}'
    if sd is not None and pd.notna(sd) and (n is None or n > 1):
        text += f' ± {sd:.{decimals}f}'
    return text

def export_latex_table(frame, name, *, caption, label, alignment=None, rules_after=(), size='small'):
    """Export the requested table* convention. Input cells/caption are plain text.

    rules_after contains zero-based row positions for additional horizontal rules.
    Numeric provenance (seed IDs, n, unrounded mean/SD) belongs in a separate CSV.
    """
    if not re.fullmatch('[A-Za-z0-9_-]+', name):
        raise ValueError('Invalid export name.')
    if not re.fullmatch('[A-Za-z0-9:_-]+', label):
        raise ValueError('Invalid LaTeX label.')
    alignment = alignment or 'l' * len(frame.columns)
    if len(alignment) != len(frame.columns) or any((c not in 'lcr' for c in alignment)):
        raise ValueError('Provide one l/c/r alignment per column.')
    if size not in ('normalsize', 'small', 'footnotesize', 'scriptsize'):
        raise ValueError('Unsupported table font size.')

    def cell(value):
        if pd.isna(value) or str(value) in ('—', '--'):
            return '--'
        return latex_escape(value).replace('±', '$\\pm$')
    lines = ['\\begin{table*}[h]', '\\caption{' + latex_escape(caption) + '}', '\\label{' + label + '}', '\\begin{center}', '\\' + size, '\\begin{tabular}{' + alignment + '}', '\\hline', ' & '.join(('\\textbf{' + latex_escape(c) + '}' for c in frame.columns)) + ' \\\\', '\\hline']
    for i, row in enumerate(frame.itertuples(index=False, name=None)):
        lines.append(' & '.join((cell(value) for value in row)) + ' \\\\')
        if i in rules_after:
            lines.append('\\hline')
    if lines[-1] != '\\hline':
        lines.append('\\hline')
    lines += ['\\end{tabular}', '\\end{center}', '\\end{table*}', '']
    tex = '\n'.join(lines)
    (OUT / 'tables' / f'{name}.tex').write_text(tex, encoding='utf-8')
    frame.to_csv(OUT / 'tables' / f'{name}.csv', index=False)
    return tex


In [ ]:
from sklearn.metrics import roc_auc_score
M1_SEEDS = {'mimic': [12345, 23451, 34512, 45123, 51234], 'code15': [12345, 23451]}
M1_COHORTS = {'mimic': ['500', '1000', '2000', '4000', '8000', '16000', '50000', 'full'], 'code15': ['1000', '8000', 'full']}
M1_LEADS = ['I', 'II', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

def mortality1_specs(site):
    return [('ecgfounder_frozen', 'ecgfounder', 'frozen', 'ECGFounder frozen'), ('ecgfounder_finetuned', 'ecgfounder', 'finetuned', 'ECGFounder FT'), (f'pretrained_{site}_frozen', site, 'frozen', 'biomECG-1k frozen'), (f'pretrained_{site}_finetuned', site, 'finetuned', 'biomECG-1k FT'), ('scratch', 'scratch', 'scratch', 'biomECG scratch')]
m1_values = snapshot('mortality_same_dataset_late8_per_seed')
m1_summary = snapshot('mortality_same_dataset_late8_summary')
m1_full_sizes = json.loads((ARTIFACTS / 'cohort_sizes.json').read_text())


In [ ]:
m1_axes_width = PAPER_WIDTH_IN * (0.995 - 0.1) / (4 + 3 * 0.1)
m1_axes_height = 2.15 * (0.88 - 0.2)
m1_left, m1_right = (PAPER_WIDTH_IN * 0.1, PAPER_WIDTH_IN * 0.005)
m1_bottom, m1_top = (0.95, 2.15 * (1 - 0.88))
m1_width = m1_left + 2.1 * m1_axes_width + m1_right
m1_height = m1_bottom + m1_axes_height + m1_top
m1_fig, m1_axes = paper_subplots(1, 2, width=m1_width, height=m1_height, sharey=True, layout=None)
m1_fig.subplots_adjust(left=m1_left / m1_width, right=1 - m1_right / m1_width, bottom=m1_bottom / m1_height, top=1 - m1_top / m1_height, wspace=0.1)
for ax, (site, title) in zip(m1_axes, [('mimic', 'MIMIC'), ('code15', 'CODE-15')]):
    order = M1_COHORTS[site]
    x = training_size_axis(ax, order, full_n=m1_full_sizes[site], log=True)
    for architecture, source, regime, label in mortality1_specs(site):
        q = m1_summary[(m1_summary.site == site) & (m1_summary.architecture == architecture)]
        stats = q.set_index('cohort_label').reindex(order)
        counts = q.n.astype(int)
        count_text = str(counts.iloc[0]) if len(counts) and counts.nunique() == 1 else f'{counts.min()}–{counts.max()}' if len(counts) else '0'
        legend_label = label if len(q) else f'{label} (unavailable)'
        style = curve_style(source, fusion='late', regime=regime)
        ax.plot(x, stats['mean'], label=legend_label, **style)
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=style['color'], alpha=0.1, linewidth=0)
    ax.set_title(title, fontsize=8, fontweight='bold', y=0.985, pad=-3)
    ax.set_ylabel('')
    ax.set_xlabel('')
    ax.tick_params(labelsize=7, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.36), fontsize=5.5, handlelength=1.5, handletextpad=0.3, borderpad=0.25, labelspacing=0.2, frameon=False)
m1_fig.supxlabel('Downstream training size (patients)', fontsize=8, fontweight='bold', y=(m1_bottom - 0.38) / m1_height)
m1_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.025)
save_paper_figure(m1_fig, 'fig01_mortality_same_dataset_late8')
m1_caption = 'Mortality prediction with eight-lead late fusion on MIMIC and CODE-15. biomECG-1k pretraining uses the target dataset. Points and bands show mean test AUROC and ±1 SD. CODE-15: 2 seeds. Training size is logarithmic; the 100,000-patient cohort is excluded.'
m1_figure_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', f'\\includegraphics[width={m1_width / PAPER_WIDTH_IN:.6f}\\textwidth]{{figures/fig01_mortality_same_dataset_late8.pdf}}', '\\end{center}', '\\caption{' + latex_escape(m1_caption) + '}', '\\label{fig:mortality-same-dataset-late8}', '\\end{figure}', ''])
(OUT / 'figures/fig01_mortality_same_dataset_late8.tex').write_text(m1_figure_latex, encoding='utf-8')
print(m1_figure_latex)


In [ ]:
m2_axes_width = PAPER_WIDTH_IN * (0.995 - 0.1) / (4 + 3 * 0.1)
m2_axes_height = 2.15 * (0.88 - 0.2)
m2_left, m2_right = (PAPER_WIDTH_IN * 0.1, PAPER_WIDTH_IN * 0.005)
m2_bottom, m2_top = (2.15 * 0.2, 2.15 * (1 - 0.88))
m2_width = m2_left + 2.1 * m2_axes_width + m2_right
m2_height = m2_bottom + m2_axes_height + m2_top
m2_fig, m2_axes = paper_subplots(1, 2, width=m2_width, height=m2_height, sharey=True, layout=None)
m2_fig.subplots_adjust(left=m2_left / m2_width, right=1 - m2_right / m2_width, bottom=m2_bottom / m2_height, top=1 - m2_top / m2_height, wspace=0.1)
for index, (ax, site) in enumerate(zip(m2_axes, ['mimic', 'code15'])):
    order = M1_COHORTS[site]
    x = training_size_axis(ax, order, full_n=m1_full_sizes[site], log=True)
    for architecture, source, regime, label in mortality1_specs(site):
        if regime != 'frozen':
            continue
        q = m1_summary[(m1_summary.site == site) & (m1_summary.architecture == architecture)]
        stats = q.set_index('cohort_label').reindex(order)
        style = curve_style(source, fusion='late', regime='frozen')
        ax.plot(x, stats['mean'], **style, label='ECGFounder' if source == 'ecgfounder' else 'biomECG-1k')
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=style['color'], alpha=0.1, linewidth=0)
    ax.set_title(f'({chr(97 + index)}) ' + ('MIMIC' if site == 'mimic' else 'CODE-15'), fontsize=8, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    if index == 0:
        ax.legend(loc='lower right', fontsize=6.5, handlelength=1.7, handletextpad=0.4, borderpad=0.4, labelspacing=0.3)
m2_fig.supxlabel('Downstream training size (patients)', fontsize=8, fontweight='bold', y=0.035)
m2_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.025)
save_paper_figure(m2_fig, 'fig02_mortality_frozen_late')
m2_caption = 'Mortality prediction with frozen encoders and eight-lead late fusion on MIMIC and CODE-15. biomECG-1k uses biometric pretraining on 1,359 patients from the target dataset. Curves and bands show mean test AUROC and one SD (CODE-15: 2 seeds).'
m2_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', f'\\includegraphics[width={m2_width / PAPER_WIDTH_IN:.6f}\\textwidth]{{figures/fig02_mortality_frozen_late.pdf}}', '\\end{center}', '\\caption{' + latex_escape(m2_caption) + '}', '\\label{fig:mortality-frozen-late}', '\\end{figure}', ''])
(OUT / 'figures/fig02_mortality_frozen_late.tex').write_text(m2_latex, encoding='utf-8')
print(m2_latex)


In [ ]:
FUSION_ARCHS = ['scratch', 'pretrained_mimic_frozen', 'pretrained_mimic_finetuned', 'ecgfounder_frozen', 'ecgfounder_finetuned']
fusion_order = M1_COHORTS['mimic']
fusion_values = snapshot('mortality_mimic_early_late_per_seed')
fusion_summary = snapshot('mortality_mimic_early_late_summary')


In [ ]:
fusion_fig, fusion_axes = paper_subplots(1, 5, width=PAPER_WIDTH_IN, height=2.15, sharex=True, sharey=True, layout=None)
fusion_fig.subplots_adjust(left=0.085, right=0.995, bottom=0.2, top=0.88, wspace=0.1)
fusion_panels = [('biomECG — scratch', 'scratch'), ('biomECG-1k — frozen', 'pretrained_mimic_frozen'), ('biomECG-1k — FT', 'pretrained_mimic_finetuned'), ('ECGFounder — frozen', 'ecgfounder_frozen'), ('ECGFounder — FT', 'ecgfounder_finetuned')]
fusion_regime_names = {'scratch': 'Scratch', 'frozen': 'Frozen', 'finetuned': 'FT'}
for panel_index, (ax, (title, architecture)) in enumerate(zip(fusion_axes, fusion_panels)):
    x = training_size_axis(ax, fusion_order, full_n=m1_full_sizes['mimic'], log=True)
    source = 'scratch' if architecture == 'scratch' else 'ecgfounder' if architecture.startswith('ecgfounder') else 'mimic'
    regime = 'scratch' if architecture == 'scratch' else architecture.rsplit('_', 1)[1]
    for fusion in ['Late', 'Early']:
        q = fusion_summary[(fusion_summary.architecture == architecture) & (fusion_summary.fusion == fusion)]
        stats = q.set_index('cohort_label').reindex(fusion_order)
        counts = sorted(set(map(int, q.n)))
        count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}–{counts[-1]}' if counts else '0'
        style = curve_style(source, fusion=fusion, regime=regime)
        ax.plot(x, stats['mean'], label=fusion, **style)
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=style['color'], alpha=0.1, linewidth=0)
    model_title, regime_title = title.split(' — ')
    ax.set_title(f'$\\mathbf{{({chr(97 + panel_index)})}}$ {model_title}\n{regime_title}', fontsize=7.5, y=0.985, pad=-3, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=panel_index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    if panel_index == 0:
        ax.legend(loc='lower right', fontsize=6.5, handlelength=1.7, handletextpad=0.4, borderpad=0.4, labelspacing=0.3, borderaxespad=0.4)
fusion_fig.supxlabel('Downstream training size (patients)', fontsize=9, fontweight='bold', y=0.035)
fusion_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.025)
save_paper_figure(fusion_fig, 'fig03_mortality_mimic_early_late')
fusion_counts = []
for architecture in FUSION_ARCHS:
    name = 'biomECG scratch' if architecture == 'scratch' else ('ECGFounder ' if architecture.startswith('ecgfounder') else 'biomECG-1k ') + fusion_regime_names[architecture.rsplit('_', 1)[1]]
    for fusion in ['Late', 'Early']:
        q = fusion_summary[(fusion_summary.architecture == architecture) & (fusion_summary.fusion == fusion)]
        counts = sorted(set(map(int, q.n)))
        count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}--{counts[-1]}' if counts else '0'
        fusion_counts.append(f'{name} {fusion.lower()}: {count_text}')
fusion_caption = 'Early versus late fusion for mortality prediction on MIMIC. Each panel shows one model and training regime. Early uses 8 leads for biomECG-1k and 12 for ECGFounder; late uses 8 for both. Curves and bands show mean test AUROC ±1 SD.'
fusion_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig03_mortality_mimic_early_late.pdf}', '\\end{center}', '\\caption{' + latex_escape(fusion_caption) + '}', '\\label{fig:mortality-mimic-early-late}', '\\end{figure}', ''])
(OUT / 'figures/fig03_mortality_mimic_early_late.tex').write_text(fusion_latex, encoding='utf-8')
print(fusion_latex)


In [ ]:
ft_fig, ft_axes = paper_subplots(1, 4, width=PAPER_WIDTH_IN, height=2.15, sharex=True, sharey=True, layout=None)
ft_fig.subplots_adjust(left=0.1, right=0.995, bottom=0.2, top=0.88, wspace=0.1)
ft_panels = [('biomECG-1k', 'Early', 'pretrained_mimic'), ('biomECG-1k', 'Late', 'pretrained_mimic'), ('ECGFounder', 'Early', 'ecgfounder'), ('ECGFounder', 'Late', 'ecgfounder')]
ft_names = {'frozen': 'Frozen', 'finetuned': 'Fine-tuned'}
ft_count_notes = []
for index, (ax, (family, fusion, prefix)) in enumerate(zip(ft_axes, ft_panels)):
    x = training_size_axis(ax, fusion_order, full_n=m1_full_sizes['mimic'], log=True)
    for regime in ['frozen', 'finetuned']:
        architecture = f'{prefix}_{regime}'
        q = fusion_summary[(fusion_summary.architecture == architecture) & (fusion_summary.fusion == fusion)]
        stats = q.set_index('cohort_label').reindex(fusion_order)
        style = curve_style('ecgfounder' if family == 'ECGFounder' else 'mimic', fusion=fusion, regime=regime)
        color = style['color']
        ax.plot(x, stats['mean'], **style, label=ft_names[regime])
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=color, alpha=0.1, linewidth=0)
        counts = sorted(set(map(int, q.n)))
        count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}--{counts[-1]}' if counts else '0'
        ft_count_notes.append(f'{family} {fusion.lower()} {ft_names[regime].lower()}: {count_text}')
    ax.set_title(f'$\\mathbf{{({chr(97 + index)})}}$ {family}\n{fusion.lower()} fusion', fontsize=8, y=0.985, pad=-3, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    if index == 0:
        ax.legend(loc='lower right', fontsize=6.5, handlelength=1.7, handletextpad=0.4, borderpad=0.4, labelspacing=0.3, borderaxespad=0.4)
ft_fig.supxlabel('Downstream training size (patients)', fontsize=9, fontweight='bold', y=0.035)
ft_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.014)
save_paper_figure(ft_fig, 'fig04_mortality_mimic_frozen_vs_finetuned')
ft_architectures = ['pretrained_mimic_frozen', 'pretrained_mimic_finetuned', 'ecgfounder_frozen', 'ecgfounder_finetuned']
ft_summary = fusion_summary[fusion_summary.architecture.isin(ft_architectures)].copy()
ft_summary.to_csv(OUT / 'tables/mortality_mimic_frozen_vs_finetuned_summary.csv', index=False)
ft_values = fusion_values[fusion_values.architecture.isin(ft_architectures)].copy()
ft_values.to_csv(OUT / 'tables/mortality_mimic_frozen_vs_finetuned_per_seed.csv', index=False)
ft_pairs = ft_values.pivot(index=['site', 'seed', 'cohort_label', 'source', 'fusion'], columns='regime', values='test_patient_auc').dropna(subset=['frozen', 'finetuned']).reset_index()
ft_pairs['frozen_minus_finetuned'] = ft_pairs.frozen - ft_pairs.finetuned
ft_pairs.to_csv(OUT / 'tables/mortality_mimic_frozen_minus_finetuned_paired.csv', index=False)
ft_caption = 'Frozen versus fine-tuned encoders for mortality prediction on MIMIC, by model and fusion strategy. Early uses 8 leads for biomECG-1k and 12 for ECGFounder; late uses 8 for both. Curves and bands show mean test AUROC ±1 SD.'
ft_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig04_mortality_mimic_frozen_vs_finetuned.pdf}', '\\end{center}', '\\caption{' + latex_escape(ft_caption) + '}', '\\label{fig:mortality-mimic-frozen-finetuned}', '\\end{figure}', ''])
(OUT / 'figures/fig04_mortality_mimic_frozen_vs_finetuned.tex').write_text(ft_latex, encoding='utf-8')
print(ft_latex)


In [ ]:
combined_fig = plt.figure(figsize=(PAPER_WIDTH_IN, 4.1))
combined_top = combined_fig.add_gridspec(1, 5, left=0.085, right=0.995, bottom=0.585, top=0.94, wspace=0.1)
combined_bottom = combined_fig.add_gridspec(1, 4, left=0.1, right=0.995, bottom=0.11, top=0.465, wspace=0.1)
fusion_axes = [combined_fig.add_subplot(combined_top[0, 0])]
for column in range(1, 5):
    fusion_axes.append(combined_fig.add_subplot(combined_top[0, column], sharey=fusion_axes[0]))
ft_axes = [combined_fig.add_subplot(combined_bottom[0, column], sharey=fusion_axes[0]) for column in range(4)]
fusion_panels = [('biomECG — scratch', 'scratch'), ('biomECG-1k — frozen', 'pretrained_mimic_frozen'), ('biomECG-1k — FT', 'pretrained_mimic_finetuned'), ('ECGFounder — frozen', 'ecgfounder_frozen'), ('ECGFounder — FT', 'ecgfounder_finetuned')]
fusion_regime_names = {'scratch': 'Scratch', 'frozen': 'Frozen', 'finetuned': 'FT'}
for panel_index, (ax, (title, architecture)) in enumerate(zip(fusion_axes, fusion_panels)):
    x = training_size_axis(ax, fusion_order, full_n=m1_full_sizes['mimic'], log=True)
    source = 'scratch' if architecture == 'scratch' else 'ecgfounder' if architecture.startswith('ecgfounder') else 'mimic'
    regime = 'scratch' if architecture == 'scratch' else architecture.rsplit('_', 1)[1]
    for fusion in ['Late', 'Early']:
        q = fusion_summary[(fusion_summary.architecture == architecture) & (fusion_summary.fusion == fusion)]
        stats = q.set_index('cohort_label').reindex(fusion_order)
        counts = sorted(set(map(int, q.n)))
        count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}–{counts[-1]}' if counts else '0'
        style = curve_style(source, fusion=fusion, regime=regime)
        ax.plot(x, stats['mean'], label=fusion, **style)
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=style['color'], alpha=0.1, linewidth=0)
    model_title, regime_title = title.split(' — ')
    ax.set_title(f'$\\mathbf{{({chr(97 + panel_index)})}}$ {model_title}\n{regime_title}', fontsize=7.5, y=0.985, pad=-3, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=panel_index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    if panel_index == 0:
        ax.legend(loc='lower right', fontsize=6.5, handlelength=1.7, handletextpad=0.4, borderpad=0.4, labelspacing=0.3, borderaxespad=0.4)
ft_panels = [('biomECG-1k', 'Early', 'pretrained_mimic'), ('biomECG-1k', 'Late', 'pretrained_mimic'), ('ECGFounder', 'Early', 'ecgfounder'), ('ECGFounder', 'Late', 'ecgfounder')]
ft_names = {'frozen': 'Frozen', 'finetuned': 'Fine-tuned'}
ft_count_notes = []
for index, (ax, (family, fusion, prefix)) in enumerate(zip(ft_axes, ft_panels)):
    x = training_size_axis(ax, fusion_order, full_n=m1_full_sizes['mimic'], log=True)
    for regime in ['frozen', 'finetuned']:
        architecture = f'{prefix}_{regime}'
        q = fusion_summary[(fusion_summary.architecture == architecture) & (fusion_summary.fusion == fusion)]
        stats = q.set_index('cohort_label').reindex(fusion_order)
        style = curve_style('ecgfounder' if family == 'ECGFounder' else 'mimic', fusion=fusion, regime=regime)
        color = style['color']
        ax.plot(x, stats['mean'], **style, label=ft_names[regime])
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=color, alpha=0.1, linewidth=0)
        counts = sorted(set(map(int, q.n)))
        count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}--{counts[-1]}' if counts else '0'
        ft_count_notes.append(f'{family} {fusion.lower()} {ft_names[regime].lower()}: {count_text}')
    ax.set_title(f'$\\mathbf{{({chr(102 + index)})}}$ {family}\n{fusion.lower()} fusion', fontsize=8, y=0.985, pad=-3, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    if index == 0:
        ax.legend(loc='lower right', fontsize=6.5, handlelength=1.7, handletextpad=0.4, borderpad=0.4, labelspacing=0.3, borderaxespad=0.4)
combined_fig.supxlabel('Downstream training size (patients)', fontsize=9, fontweight='bold', y=0.025)
combined_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.014)
save_paper_figure(combined_fig, 'fig03_04_mortality_mimic_fusion_regimes')
combined_caption = 'Mortality prediction on MIMIC. Top (a--e): early versus late fusion, by model and downstream training strategy. Bottom (f--i): frozen versus fine-tuned encoders, by model and fusion strategy. Curves and bands show mean test AUROC and one SD.'
combined_tex = '\n'.join(['\\begin{figure}[h]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig03_04_mortality_mimic_fusion_regimes.pdf}', '\\end{center}', '\\caption{' + latex_escape(combined_caption) + '}', '\\label{fig:mortality-mimic-fusion-regimes}', '\\end{figure}', ''])
(OUT / 'figures/fig03_04_mortality_mimic_fusion_regimes.tex').write_text(combined_tex, encoding='utf-8')
print(combined_tex)


In [ ]:
late_fig, late_ax = paper_subplots(1, 1, width=PAPER_WIDTH_IN, height=2.8)
late_specs = [('scratch', 'biomECG scratch', 'scratch', '^'), ('pretrained_mimic_frozen', 'biomECG-1k frozen', 'frozen', 'o'), ('pretrained_mimic_finetuned', 'biomECG-1k FT', 'finetuned', 's'), ('ecgfounder_frozen', 'ECGFounder frozen', 'frozen', 'o'), ('ecgfounder_finetuned', 'ECGFounder FT', 'finetuned', 's')]
late_overview_summary = fusion_summary[fusion_summary.fusion.eq('Late')].copy()
x = training_size_axis(late_ax, fusion_order, full_n=m1_full_sizes['mimic'], log=True)
for architecture, label, regime, marker in late_specs:
    stats = late_overview_summary[late_overview_summary.architecture.eq(architecture)].set_index('cohort_label').reindex(fusion_order)
    family = 'ecgfounder' if architecture.startswith('ecgfounder_') else 'resnet'
    style = curve_style('ecgfounder' if family == 'ecgfounder' else 'scratch' if regime == 'scratch' else 'mimic', fusion='late', regime=regime)
    color = style['color']
    late_ax.plot(x, stats['mean'], **style, label=label)
    late_ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=color, alpha=0.1, linewidth=0)
late_ax.set_title('MIMIC — late fusion', y=0.985, pad=-3)
late_ax.set_ylabel('Test AUROC')
late_ax.set_ylim(bottom=0.75)
late_ax.legend(loc='lower right', fontsize=8, handlelength=2.2, borderpad=0.5)
save_paper_figure(late_fig, 'fig05_mortality_mimic_late_overview')
late_overview_summary.to_csv(OUT / 'tables/mortality_mimic_late_overview_summary.csv', index=False)
late_overview_counts = []
for architecture, label, regime, marker in late_specs:
    q = late_overview_summary[late_overview_summary.architecture.eq(architecture)]
    counts = sorted(set(map(int, q.n)))
    count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}--{counts[-1]}' if counts else '0'
    late_overview_counts.append(f'{label}: {count_text}')
late_overview_caption = 'Mortality prediction on MIMIC with eight-lead late fusion: biomECG scratch, frozen and fine-tuned biomECG-1k, and ECGFounder frozen and fine-tuned. biomECG-1k pretraining uses MIMIC. Curves and bands show mean test AUROC ±1 SD. The vertical axis starts at 0.75.'
late_overview_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig05_mortality_mimic_late_overview.pdf}', '\\end{center}', '\\caption{' + latex_escape(late_overview_caption) + '}', '\\label{fig:mortality-mimic-late-overview}', '\\end{figure}', ''])
(OUT / 'figures/fig05_mortality_mimic_late_overview.tex').write_text(late_overview_latex, encoding='utf-8')
print(late_overview_latex)


In [ ]:
frozen_ref_fig = plt.figure(figsize=(PAPER_WIDTH_IN, 2.3))
frozen_ref_grid = frozen_ref_fig.add_gridspec(1, 5, left=0.085, right=0.995, bottom=0.19, top=0.88, wspace=0.18)
frozen_ref_axes = [frozen_ref_fig.add_subplot(frozen_ref_grid[0, 0])]
frozen_ref_axes.append(frozen_ref_fig.add_subplot(frozen_ref_grid[0, 1], sharex=frozen_ref_axes[0], sharey=frozen_ref_axes[0]))
frozen_ref_axes.append(frozen_ref_fig.add_subplot(frozen_ref_grid[0, 2:5], sharex=frozen_ref_axes[0], sharey=frozen_ref_axes[0]))
frozen_ref_specs = [('scratch', 'biomECG scratch', 'scratch', '^'), ('pretrained_mimic_frozen', 'biomECG-1k frozen', 'frozen', 'o'), ('pretrained_mimic_finetuned', 'biomECG-1k FT', 'finetuned', 'o'), ('ecgfounder_frozen', 'ECGFounder frozen', 'frozen', 's'), ('ecgfounder_finetuned', 'ECGFounder FT', 'finetuned', 's')]
frozen_ref_selections = [frozen_ref_specs[1:3], frozen_ref_specs[3:5], [frozen_ref_specs[1], frozen_ref_specs[3]]]
frozen_ref_titles = ['$\\mathbf{(a)}$ biomECG-1k\nLate fusion', '$\\mathbf{(b)}$ ECGFounder\nLate fusion', '$\\mathbf{(c)}$ Frozen encoders — late fusion']
frozen_ref_overview_summary = fusion_summary[fusion_summary.fusion.eq('Late')].copy()
for panel, (ax, specs, title) in enumerate(zip(frozen_ref_axes, frozen_ref_selections, frozen_ref_titles)):
    x = training_size_axis(ax, fusion_order, full_n=m1_full_sizes['mimic'], log=True)
    for architecture, label, regime, marker in specs:
        q = frozen_ref_overview_summary[frozen_ref_overview_summary.architecture.eq(architecture)]
        stats = q.set_index('cohort_label').reindex(fusion_order)
        family = 'ecgfounder' if architecture.startswith('ecgfounder_') else 'resnet'
        style = curve_style('ecgfounder' if family == 'ecgfounder' else 'scratch' if regime == 'scratch' else 'mimic', fusion='late', regime=regime)
        color = style['color']
        curve_label = label if panel == 2 else 'Frozen' if regime == 'frozen' else 'FT'
        ax.plot(x, stats['mean'], **style, label=curve_label)
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=color, alpha=0.1, linewidth=0)
    ax.set_title(title, fontsize=7.5 if panel < 2 else 8, y=0.985, pad=-3, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=panel == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    ax.legend(loc='lower right', fontsize=6 if panel < 2 else 6.5, handlelength=1.5, handletextpad=0.35, borderpad=0.35, labelspacing=0.3, borderaxespad=0.35)
frozen_ref_axes[0].set_ylim(bottom=0.75)
frozen_ref_fig.supxlabel('Downstream training size (patients)', fontsize=9, fontweight='bold', y=0.035)
frozen_ref_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.025)
save_paper_figure(frozen_ref_fig, 'fig06_mortality_mimic_frozen_reference')
frozen_ref_overview_summary[frozen_ref_overview_summary.architecture.ne('scratch')].to_csv(OUT / 'tables/mortality_mimic_frozen_reference_summary.csv', index=False)
frozen_ref_overview_counts = []
for architecture, label, regime, marker in frozen_ref_specs[1:]:
    q = frozen_ref_overview_summary[frozen_ref_overview_summary.architecture.eq(architecture)]
    counts = sorted(set(map(int, q.n)))
    count_text = str(counts[0]) if len(counts) == 1 else f'{counts[0]}--{counts[-1]}' if counts else '0'
    frozen_ref_overview_counts.append(f'{label}: {count_text}')
frozen_ref_overview_caption = 'Eight-lead late fusion for mortality prediction on MIMIC. Left: frozen versus fine-tuned biomECG-1k and ECGFounder. Right: comparison of their frozen encoders. biomECG-1k pretraining uses MIMIC. Curves and bands show mean test AUROC ±1 SD; the vertical axis starts at 0.75.'
frozen_ref_overview_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig06_mortality_mimic_frozen_reference.pdf}', '\\end{center}', '\\caption{' + latex_escape(frozen_ref_overview_caption) + '}', '\\label{fig:mortality-mimic-frozen-reference}', '\\end{figure}', ''])
(OUT / 'figures/fig06_mortality_mimic_frozen_reference.tex').write_text(frozen_ref_overview_latex, encoding='utf-8')
print(frozen_ref_overview_latex)


In [ ]:
PRETRAIN_SEEDS = [12345, 23451]
PRETRAIN_SIZES = [100, 500, 1359, 8000, 16000, 50000]
PRETRAIN_ARCHS = ['pretrained_mimic_frozen', 'pretrained_mimic_finetuned']
PRETRAIN_FOUNDER_ARCHS = ['ecgfounder_frozen', 'ecgfounder_finetuned']
pretrain_full_n = m1_full_sizes['mimic']
pretrain_values = snapshot('mortality_full_pretraining_size_per_seed')
pretrain_summary = snapshot('mortality_full_pretraining_size_summary')
pretrain_founder_values = snapshot('mortality_full_pretraining_size_founder_per_seed')
pretrain_founder_summary = snapshot('mortality_full_pretraining_size_founder_summary')


In [ ]:
pretrain_axes_width = PAPER_WIDTH_IN * (0.995 - 0.1) / (4 + 3 * 0.1)
pretrain_axes_height = 2.15 * (0.88 - 0.2)
pretrain_left, pretrain_right = (0.43, 0.06)
pretrain_bottom, pretrain_top = (0.43, 0.26)
pretrain_width = pretrain_left + pretrain_axes_width + pretrain_right
pretrain_height = pretrain_bottom + pretrain_axes_height + pretrain_top
pretrain_fig, pretrain_ax = paper_subplots(1, 1, width=pretrain_width, height=pretrain_height, layout=None)
pretrain_fig.subplots_adjust(left=pretrain_left / pretrain_width, right=1 - pretrain_right / pretrain_width, bottom=pretrain_bottom / pretrain_height, top=1 - pretrain_top / pretrain_height)
x = training_size_axis(pretrain_ax, PRETRAIN_SIZES, log=True)
for architecture, regime, label in [('pretrained_mimic_frozen', 'frozen', 'biomECG frozen'), ('pretrained_mimic_finetuned', 'finetuned', 'biomECG FT')]:
    values = pretrain_summary[pretrain_summary.architecture.eq(architecture)].set_index('pretraining_size').reindex(PRETRAIN_SIZES)
    style = curve_style('mimic', fusion='late', regime=regime)
    color = style['color']
    pretrain_ax.plot(x, values['mean'], **style, label=label)
    pretrain_ax.fill_between(x, values['mean'] - values['sd'], values['mean'] + values['sd'], color=color, alpha=0.1, linewidth=0)
for architecture, regime, label in [('ecgfounder_frozen', 'frozen', 'ECGFounder frozen'), ('ecgfounder_finetuned', 'finetuned', 'ECGFounder FT')]:
    reference = pretrain_founder_summary[pretrain_founder_summary.architecture.eq(architecture)].iloc[0]
    reference_style = curve_style('ecgfounder', fusion='late', regime=regime)
    reference_style.update(linestyle=':' if regime == 'frozen' else '-', marker=None)
    pretrain_ax.axhline(reference['mean'], **reference_style, label=label)
pretrain_ax.set_title('MIMIC Full\nPretraining size', fontsize=7.5, fontweight='bold', y=0.985, pad=-3)
pretrain_ax.set_xlabel('Pretraining size (patients)', fontsize=7, fontweight='bold', labelpad=1.5)
pretrain_ax.set_ylabel('Test AUROC', fontsize=7, fontweight='bold', labelpad=1.5)
pretrain_ax.tick_params(labelsize=6.5, length=2.5, pad=2)
pretrain_ax.tick_params(which='minor', length=1.5)
pretrain_ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
pretrain_ax.legend(loc='lower right', fontsize=6.5, handlelength=1.3, handletextpad=0.3, borderpad=0.3, labelspacing=0.3, borderaxespad=0.25)
save_paper_figure(pretrain_fig, 'fig07_mortality_full_pretraining_size')
pretrain_caption = 'Mortality AUROC on the full MIMIC downstream cohort versus biometric pretraining size (train + validation patients). Eight-lead late fusion; 2 seeds. biomECG curves show mean AUROC with shaded bands of one SD; horizontal lines show mean ECGFounder references.'
pretrain_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', f'\\includegraphics[width={pretrain_width / PAPER_WIDTH_IN:.6f}\\textwidth]{{figures/fig07_mortality_full_pretraining_size.pdf}}', '\\end{center}', '\\caption{' + latex_escape(pretrain_caption) + '}', '\\label{fig:mortality-full-pretraining-size}', '\\end{figure}', ''])
(OUT / 'figures/fig07_mortality_full_pretraining_size.tex').write_text(pretrain_latex, encoding='utf-8')
print(pretrain_latex)


In [ ]:
cross_values = snapshot('mortality_mimic_code15_comparison_code15_per_seed')
cross_summary = snapshot('mortality_mimic_code15_comparison_code15_summary')
cross_code15_full = m1_full_sizes['code15']
cross_fig, cross_axes = paper_subplots(1, 4, width=PAPER_WIDTH_IN, height=2.15, sharey=True, layout=None)
cross_fig.subplots_adjust(left=0.1, right=0.995, bottom=0.2, top=0.88, wspace=0.1)
cross_panels = [('MIMIC', 'biomECG-1k', ['pretrained_mimic_frozen', 'pretrained_mimic_finetuned']), ('MIMIC', 'ECGFounder', ['ecgfounder_frozen', 'ecgfounder_finetuned']), ('MIMIC', 'Frozen encoders', ['pretrained_mimic_frozen', 'ecgfounder_frozen']), ('CODE-15', 'Frozen encoders', ['pretrained_code15_frozen', 'ecgfounder_frozen'])]
for index, (ax, (site, title, architectures)) in enumerate(zip(cross_axes, cross_panels)):
    order = fusion_order if site == 'MIMIC' else ['1000', '8000', 'full']
    full_n = m1_full_sizes['mimic'] if site == 'MIMIC' else cross_code15_full
    x = training_size_axis(ax, order, full_n=full_n, log=True)
    source = fusion_summary[fusion_summary.fusion.eq('Late')] if site == 'MIMIC' else cross_summary
    for architecture in architectures:
        stats = source[source.architecture.eq(architecture)].set_index('cohort_label').reindex(order)
        founder = architecture.startswith('ecgfounder')
        regime = architecture.rsplit('_', 1)[1]
        style = curve_style('ecgfounder' if founder else 'mimic', fusion='late', regime=regime)
        color = style['color']
        label = ('Frozen' if regime == 'frozen' else 'FT') if index < 2 else 'ECGFounder' if founder else 'biomECG-1k'
        ax.plot(x, stats['mean'], **style, label=label)
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=color, alpha=0.1, linewidth=0)
    ax.set_title(f'({chr(97 + index)}) {site}\n{title}', fontsize=7.5, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    ax.legend(loc='lower right', fontsize=6, handlelength=1.5, handletextpad=0.35, borderpad=0.35, labelspacing=0.3, borderaxespad=0.35)
cross_axes[0].set_ylim(bottom=0.7)
cross_fig.supxlabel('Downstream training size (patients)', fontsize=9, fontweight='bold', y=0.035)
cross_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.014)
save_paper_figure(cross_fig, 'fig08_mortality_mimic_code15_frozen')
cross_full = cross_summary[cross_summary.cohort_label.eq('full')].set_index('architecture')['mean']
cross_full_claim = all((cross_full[f'{prefix}_frozen'] > cross_full[f'{prefix}_finetuned'] for prefix in ['pretrained_code15', 'ecgfounder']))
cross_claim = 'On full CODE-15, frozen encoders outperform fine-tuning for both families within late fusion; ECGFounder fine-tuning is slightly better at smaller cohort sizes.' if cross_full_claim else 'The CODE-15 frozen versus fine-tuned ranking depends on model and cohort size.'
cross_caption = 'Eight-lead late-fusion mortality prediction: frozen versus fine-tuned encoders on MIMIC (a,b), and frozen biomECG-1k versus ECGFounder on MIMIC (c) and CODE-15 (d). Curves and bands show mean test AUROC and one SD (CODE-15: 2 seeds). ' + cross_claim
cross_latex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig08_mortality_mimic_code15_frozen.pdf}', '\\end{center}', '\\caption{' + latex_escape(cross_caption) + '}', '\\label{fig:mortality-mimic-code15-frozen}', '\\end{figure}', ''])
(OUT / 'figures/fig08_mortality_mimic_code15_frozen.tex').write_text(cross_latex, encoding='utf-8')
print(cross_latex)


In [ ]:
source_axes_width = PAPER_WIDTH_IN * (0.995 - 0.1) / (4 + 3 * 0.1)
source_axes_height = 2.15 * (0.88 - 0.2)
source_left, source_right = (PAPER_WIDTH_IN * 0.1, PAPER_WIDTH_IN * 0.005)
source_bottom, source_top = (2.15 * 0.2, 2.15 * (1 - 0.88))
source_width = source_left + 2.1 * source_axes_width + source_right
source_height = source_bottom + source_axes_height + source_top
source_fig, source_axes = paper_subplots(1, 2, width=source_width, height=source_height, sharey=True, layout=None)
source_fig.subplots_adjust(left=source_left / source_width, right=1 - source_right / source_width, bottom=source_bottom / source_height, top=1 - source_top / source_height, wspace=0.1)
for panel_index, (ax, site) in enumerate(zip(source_axes, ['mimic', 'code15'])):
    sources = [('pretrained_mimic_frozen', 'biomECG-1k pretrained MIMIC', 'mimic')]
    if site == 'code15':
        sources.append(('pretrained_code15_frozen', 'biomECG-1k pretrained CODE-15', 'code15'))
    seeds = M1_SEEDS[site]
    values = snapshot(f'frozen_source_comparison_{site}')
    order = M1_COHORTS[site]
    full_n = m1_full_sizes[site]
    x = training_size_axis(ax, order, full_n=full_n)
    for architecture, label, color_key in sources:
        stats = values[values.architecture.eq(architecture)].groupby('cohort_label').test_patient_auc.agg(['mean', 'std', 'count']).reindex(order)
        style = curve_style(color_key, fusion='late', regime='frozen')
        style['linestyle'] = SOURCE_LINESTYLES[color_key]
        color = style['color']
        ax.plot(x, stats['mean'], **style, label=label)
        ax.fill_between(x, stats['mean'] - stats['std'], stats['mean'] + stats['std'], color=color, alpha=0.1, linewidth=0)
    ax.set_title(f'({chr(97 + panel_index)}) ' + ('MIMIC' if site == 'mimic' else 'CODE-15'), fontsize=8, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(labelsize=7, labelleft=panel_index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    values.to_csv(OUT / 'tables' / f'frozen_source_comparison_{site}.csv', index=False)
source_handles = [Line2D([], [], **dict(curve_style(key, fusion='late', regime='frozen'), linestyle=SOURCE_LINESTYLES[key]), label=f'biomECG-1k pretrained {label}') for key, label in [('mimic', 'MIMIC'), ('code15', 'CODE-15')]]
source_axes[1].legend(handles=source_handles, loc='lower right', fontsize=5, frameon=True, handlelength=1.6, handletextpad=0.3, borderpad=0.3, labelspacing=0.3)
source_fig.supxlabel('Downstream training size (patients)', fontsize=8, fontweight='bold', y=0.035)
source_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.025)
save_paper_figure(source_fig, 'fig09_frozen_pretraining_sources')
caption = 'Frozen biomECG-1k mortality prediction with eight-lead late fusion, by biometric pretraining source. Curves and bands show mean test AUROC and one SD on MIMIC (left) and CODE-15 (right; 2 seeds).'
tex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', f'\\includegraphics[width={source_width / PAPER_WIDTH_IN:.6f}\\textwidth]{{figures/fig09_frozen_pretraining_sources.pdf}}', '\\end{center}', '\\caption{' + latex_escape(caption) + '}', '\\label{fig:frozen-pretraining-sources}', '\\end{figure}'])
(OUT / 'figures/fig09_frozen_pretraining_sources.tex').write_text(tex, encoding='utf-8')
print(tex)


In [ ]:
DEM_COHORT_ORDER = ['1000', '8000', 'full']
DEM_SITES = {'mimic': 'MIMIC', 'code15': 'CODE-15'}
dem_values = snapshot('demographics_complete_per_seed')
dem_summary = snapshot('demographics_complete_summary')

def dem_architectures(site):
    return ['scratch', 'ecgfounder_frozen', 'ecgfounder_finetuned', 'pretrained_frozen', 'pretrained_finetuned', 'pretrained_mimic_frozen', 'pretrained_mimic_finetuned', 'pretrained_code15_frozen', 'pretrained_code15_finetuned']

def dem_model_name(architecture, site):
    if architecture == 'scratch':
        return 'biomECG scratch'
    if architecture.startswith('ecgfounder_'):
        return 'ECGFounder'
    source = 'MIMIC' if architecture.startswith('pretrained_mimic_') else 'CODE-15' if architecture.startswith('pretrained_code15_') else 'MIMIC'
    return f'biomECG-1k ({source})'

def dem_stat(site, outcome, architecture, fusion, cohort):
    q = dem_summary[(dem_summary.site == site) & (dem_summary.task == outcome) & (dem_summary.architecture == architecture) & (dem_summary.fusion == fusion) & (dem_summary.cohort_label == cohort)]
    return None if q.empty else q.iloc[0]


In [ ]:
demo_fig, demo_axes = paper_subplots(1, 4, width=PAPER_WIDTH_IN, height=2.35, layout=None)
demo_fig.subplots_adjust(left=0.085, right=0.995, bottom=0.29, top=0.95, wspace=0.32)
demo_panels = [('age', 'mimic'), ('age', 'code15'), ('sex', 'mimic'), ('sex', 'code15')]
for index, (ax, (outcome, site)) in enumerate(zip(demo_axes, demo_panels)):
    full_n = m1_full_sizes[site]
    x = training_size_axis(ax, DEM_COHORT_ORDER, full_n=full_n, log=True)
    for architecture in ['ecgfounder_frozen', f'pretrained_{site}_frozen']:
        for fusion in ['Late', 'Early']:
            stats = [dem_stat(site, outcome, architecture, fusion, c) for c in DEM_COHORT_ORDER]
            y = np.array([np.nan if t is None else t['mean'] for t in stats])
            sd = np.array([np.nan if t is None else t['sd'] for t in stats])
            if not np.isfinite(y).any():
                continue
            style = curve_style('ecgfounder' if architecture.startswith('ecgfounder') else site, fusion=fusion, regime='frozen')
            color = style['color']
            ax.plot(x, y, **style)
            ax.fill_between(x, y - sd, y + sd, color=color, alpha=0.1, linewidth=0)
    task_label = 'Age' if outcome == 'age' else 'Sex'
    ax.set_title(f'({chr(97 + index)}) {DEM_SITES[site]} — {task_label}', fontsize=7.5, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('')
    ax.set_ylabel('MAE (years)' if index == 0 else 'AUROC' if index == 2 else '', fontsize=7, fontweight='bold', labelpad=1)
    ax.tick_params(labelsize=6.5, length=2.5, pad=1.5)
    ax.tick_params(which='minor', length=1.5)
for a, b in [(0, 1), (2, 3)]:
    limits = [demo_axes[a].get_ylim(), demo_axes[b].get_ylim()]
    for index in [a, b]:
        demo_axes[index].set_ylim(min((v[0] for v in limits)), max((v[1] for v in limits)))
    demo_axes[b].tick_params(labelleft=False)
demo_handles = [Line2D([], [], **curve_style(source, fusion=fusion, regime='frozen'), label=f'{label} frozen — {fusion}') for source, label in [('mimic', 'biomECG-1k'), ('ecgfounder', 'ECGFounder')] for fusion in ['late', 'early']]
demo_fig.supxlabel('Downstream training size (patients)', fontsize=9, fontweight='bold', y=0.025)
demo_fig.legend(handles=demo_handles, loc='lower center', bbox_to_anchor=(0.5, 0.16), ncol=4, fontsize=6.5, frameon=False, handlelength=1.8, columnspacing=1.2, borderaxespad=0, borderpad=0)
save_paper_figure(demo_fig, 'fig10_demographics_mimic_code15')
demo_caption = 'Age (MAE) and sex (AUROC) prediction on MIMIC and CODE-15 using frozen representations. Late fusion averages eight leads; early fusion uses eight for biomECG-1k and twelve for ECGFounder. Curves and bands show mean and one SD.'
demo_two_seed_sites = []
for site, site_label in DEM_SITES.items():
    subset = dem_summary[(dem_summary.site == site) & dem_summary.architecture.isin(['ecgfounder_frozen', f'pretrained_{site}_frozen'])]
    if not subset.empty and subset['n'].eq(2).all():
        demo_two_seed_sites.append(site_label)
if len(demo_two_seed_sites) == len(DEM_SITES):
    demo_caption += ' 2 seeds.'
elif demo_two_seed_sites:
    demo_caption += ' ' + ', '.join(demo_two_seed_sites) + ': 2 seeds.'
demo_tex = '\n'.join(['\\begin{figure}[t]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig10_demographics_mimic_code15.pdf}', '\\end{center}', '\\caption{' + latex_escape(demo_caption) + '}', '\\label{fig:demographics-mimic-code15}', '\\end{figure}'])
(OUT / 'figures/fig10_demographics_mimic_code15.tex').write_text(demo_tex, encoding='utf-8')
print(demo_tex)


In [ ]:
lead_paired_values = snapshot('ecgfounder_late8_vs_late12_per_seed')
lead_summary = snapshot('ecgfounder_late8_vs_late12_summary')
lead_models = ['mean_late_fusion_8_leads', 'mean_late_fusion']
lead_aw = PAPER_WIDTH_IN * (0.995 - 0.1) / (4 + 3 * 0.1)
lead_ah = 2.15 * (0.88 - 0.2)
lead_left, lead_right, lead_bottom, lead_top = (PAPER_WIDTH_IN * 0.1, PAPER_WIDTH_IN * 0.005, 0.43, 0.258)
lead_width = lead_left + 2.1 * lead_aw + lead_right
lead_height = lead_bottom + lead_ah + lead_top
lead_fig, lead_axes = paper_subplots(1, 2, width=lead_width, height=lead_height, sharey=True, layout=None)
lead_fig.subplots_adjust(left=lead_left / lead_width, right=1 - lead_right / lead_width, bottom=lead_bottom / lead_height, top=1 - lead_top / lead_height, wspace=0.1)
for index, (ax, regime) in enumerate(zip(lead_axes, ['frozen', 'finetuned'])):
    x = training_size_axis(ax, M1_COHORTS['mimic'], full_n=m1_full_sizes['mimic'])
    for model, label, line in [('mean_late_fusion_8_leads', '8 leads', '-'), ('mean_late_fusion', '12 leads', '--')]:
        stats = lead_summary[(lead_summary.architecture == f'ecgfounder_{regime}') & (lead_summary.model == model)].set_index('cohort_label').reindex(M1_COHORTS['mimic'])
        style = curve_style('ecgfounder', fusion='late', regime=regime)
        style['linestyle'] = line
        ax.plot(x, stats['mean'], **style, label=label)
        ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=style['color'], alpha=0.1, linewidth=0)
    ax.set_title(f'({chr(97 + index)}) MIMIC\n' + ('Frozen' if regime == 'frozen' else 'Fine-tuned'), fontsize=8, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7, labelleft=index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
    ax.legend(fontsize=6.5, loc='lower right', handlelength=2, borderpad=0.3)
lead_fig.supxlabel('Downstream training size (patients)', fontsize=8, fontweight='bold', y=0.025)
lead_fig.supylabel('Test AUROC', fontsize=9, fontweight='bold', x=0.025)
save_paper_figure(lead_fig, 'fig11_ecgfounder_late8_vs_late12')
lead_caption = 'ECGFounder mortality prediction on MIMIC with frozen and fine-tuned encoders. Late fusion averages either eight experts (I, II, V1--V6; solid) or all twelve (dashed). Curves and bands show mean test AUROC and one SD using matched runs at each training size.'
if lead_summary['n'].eq(2).all():
    lead_caption += ' 2 seeds.'
lead_tex = '\n'.join(['\\begin{figure}[h]', '\\begin{center}', f'\\includegraphics[width={lead_width / PAPER_WIDTH_IN:.6f}\\textwidth]{{figures/fig11_ecgfounder_late8_vs_late12.pdf}}', '\\end{center}', '\\caption{' + latex_escape(lead_caption) + '}', '\\label{fig:ecgfounder-late8-vs-late12}', '\\end{figure}', ''])
(OUT / 'figures/fig11_ecgfounder_late8_vs_late12.tex').write_text(lead_tex, encoding='utf-8')
print(lead_tex)


In [ ]:
from matplotlib.colors import BoundaryNorm
seed_mortality_cohorts = M1_COHORTS
seed_demographic_cohorts = DEM_COHORT_ORDER
seed_fused = snapshot('completed_seeds_by_experiment')
seed_counts = snapshot('seed_counts_by_experiment')
seed_runs = seed_fused
seed_arch_labels = {'scratch': 'biomECG scratch', 'ecgfounder_frozen': 'ECGF frozen', 'ecgfounder_finetuned': 'ECGF FT', 'pretrained_mimic_frozen': 'biomECG MIMIC frozen', 'pretrained_mimic_finetuned': 'biomECG MIMIC FT', 'pretrained_code15_frozen': 'biomECG CODE-15 frozen', 'pretrained_code15_finetuned': 'biomECG CODE-15 FT'}
seed_max = max(5, int(seed_counts.n.max())) if len(seed_counts) else 5
seed_cmap = plt.get_cmap('Blues', seed_max + 1)
seed_norm = BoundaryNorm(np.arange(-0.5, seed_max + 1.5), seed_max + 1)
seed_fig, seed_axes = plt.subplots(2, 3, figsize=(PAPER_WIDTH_IN, 4.6), layout='constrained')
for row, site in enumerate(['mimic', 'code15']):
    present = set(seed_runs.loc[seed_runs.site.eq(site), 'architecture'])
    rows = [(a, f) for a in seed_arch_labels if a in present and (not (site == 'mimic' and a.startswith('pretrained_code15_'))) for f in ['Early', 'Late']]
    for col, task in enumerate(['Mortality', 'Age', 'Sex']):
        ax = seed_axes[row, col]
        cohorts = seed_mortality_cohorts[site] if task == 'Mortality' else seed_demographic_cohorts
        subset = seed_counts[(seed_counts.site == site) & (seed_counts.task == task)]
        lookup = {(r.architecture, r.fusion, str(r.cohort)): int(r.n) for r in subset.itertuples()}
        matrix = np.array([[lookup.get((a, f, str(c)), 0) for c in cohorts] for a, f in rows], dtype=int)
        if not subset.empty and matrix.max() == 0:
            raise ValueError(f'Heatmap key mismatch: {site} {task}')
        im = ax.imshow(matrix, cmap=seed_cmap, norm=seed_norm, aspect='auto', interpolation='nearest')
        for (y, x), count in np.ndenumerate(matrix):
            ax.text(x, y, str(count), ha='center', va='center', fontsize=5, color='white' if count > seed_max * 0.55 else '#222222')
        ax.set_xticks(range(len(cohorts)), ['Full' if c == 'full' else f'{int(c) / 1000:g}k' for c in cohorts], rotation=90, fontsize=5.5)
        ax.set_yticks(range(len(rows)), [seed_arch_labels[a] + ' ' + ('E' if f == 'Early' else 'L') for a, f in rows], fontsize=5)
        ax.tick_params(axis='both', length=0, pad=1, labelleft=col == 0)
        ax.grid(False)
        ax.set_title(f'{('MIMIC' if site == 'mimic' else 'CODE-15')} — {task}', fontsize=7, fontweight='bold', pad=4)
        ax.set_xlabel('Downstream training size', fontsize=6, labelpad=1)
seed_fig.colorbar(im, ax=seed_axes, ticks=range(seed_max + 1), shrink=0.45, pad=0.015).ax.tick_params(labelsize=6)
save_paper_figure(seed_fig, 'fig12_seed_counts_heatmaps')
seed_caption = 'Number of completed seeds per experiment on MIMIC and CODE-15. E: early fusion; L: eight-lead late fusion, counted only when all eight experts are complete. Biometric pretraining uses 1,359 patients. Zero denotes no complete result, including configurations not run.'
seed_tex = '\n'.join(['\\begin{figure}[h]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig12_seed_counts_heatmaps.pdf}', '\\end{center}', '\\caption{' + latex_escape(seed_caption) + '}', '\\label{fig:seed-counts}', '\\end{figure}'])
(OUT / 'figures/fig12_seed_counts_heatmaps.tex').write_text(seed_tex, encoding='utf-8')
print(seed_tex)


In [ ]:
from decimal import Decimal
result_arch_names = {'scratch': ('biomECG scratch', '--'), 'ecgfounder_frozen': ('ECGFounder', 'Frozen'), 'ecgfounder_finetuned': ('ECGFounder', 'FT'), 'pretrained_mimic_frozen': ('biomECG-1k MIMIC', 'Frozen'), 'pretrained_mimic_finetuned': ('biomECG-1k MIMIC', 'FT'), 'pretrained_code15_frozen': ('biomECG-1k CODE-15', 'Frozen'), 'pretrained_code15_finetuned': ('biomECG-1k CODE-15', 'FT')}

def result_sig(value, digits):
    return format(Decimal(format(float(value), f'#.{digits}g')), 'f')

def result_entry(row):
    if row is None or not np.isfinite(row['mean']):
        return '--'
    sd = result_sig(row['sd'], 2) if row['n'] > 1 and np.isfinite(row['sd']) else '--'
    return result_sig(row['mean'], 3) + ' (' + sd + ')'
result_values = snapshot('complete_results_per_seed')
result_summary = snapshot('complete_results_mean_sd')
result_tex_all = []
for site, site_label in [('mimic', 'MIMIC'), ('code15', 'CODE-15')]:
    site_archs = [a for a in result_arch_names if a in set(result_values.loc[result_values.site.eq(site), 'architecture'])]
    for task in ['Mortality', 'Age', 'Sex']:
        cohorts = M1_COHORTS[site] if task == 'Mortality' else DEM_COHORT_ORDER
        subset = result_summary[(result_summary.site == site) & (result_summary.task == task)]
        lookup = {(r['architecture'], r['fusion'], str(r['cohort_label'])): r for _, r in subset.iterrows()}
        rows = []
        for architecture in site_archs:
            model, regime = result_arch_names[architecture]
            for fusion in ['Early', 'Late']:
                cells = [model, regime, fusion] + [result_entry(lookup.get((architecture, fusion, str(c)))) for c in cohorts]
                rows.append(' & '.join(cells) + ' \\\\')
        metric = 'MAE in years' if task == 'Age' else 'AUROC'
        caption = f'{site_label} {task.lower()} results ({metric}): mean (SD). Seed counts: Fig.~\\ref{{fig:seed-counts}}. Late fusion uses eight leads; -- indicates unavailable results.'
        headers = ['Model', 'Regime', 'Fusion'] + ['Full' if c == 'full' else f'{int(c):,}' for c in cohorts]
        tex = '\n'.join(['\\begin{table*}[h]', '\\caption{' + caption + '}', '\\label{tab:results-' + site + '-' + task.lower() + '}', '\\begin{center}', '\\scriptsize', '\\setlength{\\tabcolsep}{2pt}', '\\begin{tabular}{lll' + 'r' * len(cohorts) + '}', '\\hline', ' & '.join(('\\textbf{' + h + '}' for h in headers)) + ' \\\\', '\\hline', *rows, '\\hline', '\\end{tabular}', '\\end{center}', '\\end{table*}', ''])
        (OUT / 'tables' / f'results_{site}_{task.lower()}.tex').write_text(tex, encoding='utf-8')
        result_tex_all.append(tex)
        print(tex)
(OUT / 'tables/results_complete_appendix.tex').write_text('\n'.join(result_tex_all), encoding='utf-8')


In [ ]:
from decimal import Decimal
eer_values = snapshot('biometric_eer_per_seed')
eer_summary = snapshot('biometric_eer_summary')
eer_lead_order = ['I', 'II', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', '8l']
eer_lead_order += sorted(set(eer_values.lead) - set(eer_lead_order))

def eer_sig(value, digits):
    return format(Decimal(format(float(value), f'#.{digits}g')), 'f')

def eer_entry(row):
    if row is None:
        return '--'
    sd = eer_sig(row['sd'], 2) if row['n'] > 1 and np.isfinite(row['sd']) else '--'
    return eer_sig(row['mean'], 3) + ' (' + sd + ')'
eer_display_rows, eer_tex_rows = ([], [])
for site, site_label in [('MIMIC', 'MIMIC'), ('CODE15', 'CODE-15')]:
    subset = eer_summary[eer_summary.site == site]
    for size in sorted(subset.pretraining_size.unique()):
        stats = subset[subset.pretraining_size == size].set_index('lead')
        model = 'biomECG-1k' if size == 1359 else 'biomECG'
        prefix = [model, site_label, f'{int(size):,}']
        cells = [eer_entry(stats.loc[lead] if lead in stats.index else None) for lead in eer_lead_order]
        eer_display_rows.append(prefix + cells)
        eer_tex_rows.append(' & '.join(prefix + cells) + ' \\\\')
    eer_tex_rows.append('\\hline')
eer_columns = ['Model', 'Dataset', 'Pretraining patients'] + ['Early (8L)' if l == '8l' else l for l in eer_lead_order]
eer_table = pd.DataFrame(eer_display_rows, columns=eer_columns)
eer_table.to_csv(OUT / 'tables/biometric_eer_appendix.csv', index=False)
display(eer_table)
display(eer_summary.pivot(index=['site', 'pretraining_size'], columns='lead', values='n').reindex(columns=eer_lead_order))
eer_caption = 'Biometric verification on cross-session test pairs: EER (\\%), mean (SD) across available seeds. The equal error rate (EER) is the operating point where false acceptance and false rejection rates are equal; lower is better. Each column denotes a separately trained encoder; Early (8L) jointly encodes eight leads. Increasing the MIMIC pretraining cohort generally reduces EER, indicating a scaling benefit. -- denotes unavailable results or SD.'
eer_two_seed_groups = eer_summary.groupby(['site', 'pretraining_size'])['n'].apply(lambda x: x.eq(2).all())
eer_two_seed_labels = [('CODE-15' if site == 'CODE15' else site) + f' {int(size):,}' for (site, size), is_two in eer_two_seed_groups.items() if is_two]
if eer_two_seed_labels:
    eer_caption += ' Two seeds: ' + ', '.join(eer_two_seed_labels) + '.'
eer_tex = '\n'.join(['\\begin{table*}[h]', '\\caption{' + eer_caption + '}', '\\label{tab:biometric-eer}', '\\begin{center}', '\\scriptsize', '\\setlength{\\tabcolsep}{2pt}', '\\begin{tabular}{llr' + 'r' * len(eer_lead_order) + '}', '\\hline', ' & '.join(('\\textbf{' + h + '}' for h in eer_columns)) + ' \\\\', '\\hline', *eer_tex_rows, '\\end{tabular}', '\\end{center}', '\\end{table*}', ''])
(OUT / 'tables/biometric_eer_appendix.tex').write_text(eer_tex, encoding='utf-8')
print(eer_tex)


In [ ]:
patient_view = snapshot('umap_display_labels')
with np.load(ARTIFACTS / 'umap_coordinates.npz', allow_pickle=False) as archive:
    coordinates = [archive['view_0'], archive['view_1']]
patient_umap = dict(identity=patient_view, coordinates=coordinates, UMAP_GRID=[dict(metric='euclidean', n_neighbors=15, min_dist=0.1), dict(metric='cosine', n_neighbors=50, min_dist=0.5)])


In [ ]:
patient_view = patient_umap['identity']
patient_names = sorted(patient_view.patient_display.unique())
patient_markers = ['o', 's', '^', 'v', 'D', 'P', 'X', '*', '<', '>', 'p', 'h', 'H', '8', 'd', (3, 0, 0), (3, 0, 60), (4, 0, 45), (5, 1, 0), (6, 1, 0)]
patient_colors = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#332288', '#882255', '#44AA99', '#AA4499', '#117733', '#000000', '#56B4E9', '#A6761D', '#E31A1C', '#666666', '#1B9E77', '#7570B3', '#B2182B', '#6B8E23', '#8C510A']
patient_styles = {name: (patient_colors[i % 20], patient_markers[(i + i // 20 * 7) % 20]) for i, name in enumerate(patient_names)}
patient_fig, patient_axes = paper_subplots(1, 2, width=PAPER_WIDTH_IN, height=3.7, layout=None)
patient_fig.subplots_adjust(left=0.07, right=0.99, bottom=0.35, top=0.94, wspace=0.18)
for index, (ax, z, params) in enumerate(zip(patient_axes, patient_umap['coordinates'], patient_umap['UMAP_GRID'])):
    for name in patient_names:
        mask = patient_view.patient_display.eq(name).to_numpy()
        color, marker = patient_styles[name]
        ax.scatter(z[mask, 0], z[mask, 1], c=color, marker=marker, s=30, alpha=0.95, edgecolors='white', linewidths=0.45, rasterized=True)
    ax.set_title(f'({chr(97 + index)}) {params['metric'].capitalize()} | k={params['n_neighbors']}, d={params['min_dist']}', fontsize=8, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('UMAP 1', fontsize=8, labelpad=1)
    ax.set_ylabel('UMAP 2', fontsize=8, labelpad=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
patient_handles = [Line2D([], [], ls='', marker=patient_styles[name][1], color=patient_styles[name][0], markersize=5, markeredgecolor='white', markeredgewidth=0.4, label=name) for name in patient_names]
patient_fig.legend(handles=patient_handles, loc='lower center', bbox_to_anchor=(0.5, 0.015), ncol=8, fontsize=6, frameon=False, handletextpad=0.25, columnspacing=1.1, labelspacing=0.55)
save_paper_figure(patient_fig, 'fig13_umap_biometric_patient_identity')
patient_view.to_csv(OUT / 'tables/umap_identity_40_patients_first5.csv', index=False)
patient_caption = 'UMAP of 128-dimensional lead-II backbone features before the Siamese projection head, using biometric pretraining on 1,359 MIMIC patients. Colors and symbols identify 40 randomly selected patients with repeated recordings; up to their five earliest ECGs are shown. Patients are sampled from the available MIMIC cohort, including potentially pretraining patients. UMAP uses Euclidean distance with 15 neighbors and min_dist=0.1 (a), and cosine distance with 50 neighbors and min_dist=0.5 (b). Both projections use identical ECGs and are fitted without patient labels.'
patient_tex = '\n'.join(['\\begin{figure}[h]', '\\begin{center}', '\\includegraphics[width=\\textwidth]{figures/fig13_umap_biometric_patient_identity.pdf}', '\\end{center}', '\\caption{' + latex_escape(patient_caption) + '}', '\\label{fig:umap-biometric-patient-identity}', '\\end{figure}', ''])
(OUT / 'figures/fig13_umap_biometric_patient_identity.tex').write_text(patient_tex, encoding='utf-8')
print(patient_tex)


In [ ]:
age_bias_values = snapshot('age_late_fusion_bias_per_seed')
age_bias_summary = snapshot('age_late_fusion_bias_summary')
bias_aw = PAPER_WIDTH_IN * (0.995 - 0.1) / (4 + 3 * 0.1)
bias_ah = 2.15 * (0.88 - 0.2)
bias_left, bias_right, bias_bottom, bias_top = (PAPER_WIDTH_IN * 0.1, PAPER_WIDTH_IN * 0.005, 0.8, 0.258)
bias_width = bias_left + 2.1 * bias_aw + bias_right
bias_height = bias_bottom + bias_ah + bias_top
bias_fig, bias_axes = paper_subplots(1, 2, width=bias_width, height=bias_height, sharey=True, layout=None)
bias_fig.subplots_adjust(left=bias_left / bias_width, right=1 - bias_right / bias_width, bottom=bias_bottom / bias_height, top=1 - bias_top / bias_height, wspace=0.1)
for index, (ax, site) in enumerate(zip(bias_axes, ['mimic', 'code15'])):
    full = dem_values[(dem_values.site == site) & (dem_values.task == 'age') & dem_values.cohort_label.eq('full')]
    if full.empty:
        raise ValueError(f'Full downstream age cohort size unavailable for {site}')
    x = training_size_axis(ax, DEM_COHORT_ORDER, full_n=m1_full_sizes[site])
    ax.axhline(0, color='#888888', linewidth=0.7, linestyle=':', zorder=0)
    for source, label, prefix in [('mimic' if site == 'mimic' else 'code15', 'biomECG-1k', f'pretrained_{site}'), ('ecgfounder', 'ECGFounder', 'ecgfounder')]:
        for regime, regime_label in [('frozen', 'frozen'), ('finetuned', 'FT')]:
            stats = age_bias_summary[(age_bias_summary.site == site) & age_bias_summary.architecture.eq(f'{prefix}_{regime}')].set_index('cohort_label').reindex(DEM_COHORT_ORDER)
            style = curve_style(source, fusion='late', regime=regime)
            ax.plot(x, stats['mean'], **style, label=f'{label} {regime_label}')
            ax.fill_between(x, stats['mean'] - stats['sd'], stats['mean'] + stats['sd'], color=style['color'], alpha=0.1, linewidth=0)
    ax.set_title(f'({chr(97 + index)}) ' + ('MIMIC' if site == 'mimic' else 'CODE-15'), fontsize=8, fontweight='bold', y=0.985, pad=-3)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7, labelleft=index == 0, length=2.5, pad=2)
    ax.tick_params(which='minor', length=1.5)
bias_fig.supylabel('Age bias (years)', fontsize=9, fontweight='bold', x=0.015)
bias_fig.supxlabel('Downstream training size (patients)', fontsize=8, fontweight='bold', y=(bias_bottom - 0.38) / bias_height)
handles, labels_bias = bias_axes[0].get_legend_handles_labels()
bias_fig.legend(handles, labels_bias, loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=2, fontsize=6.5, frameon=False, handlelength=1.7, columnspacing=1)
save_paper_figure(bias_fig, 'fig14_age_late_fusion_bias')
bias_caption = 'Age prediction bias (predicted minus true age) versus downstream training size, using eight-lead late fusion with frozen or fine-tuned encoders. Positive values indicate overestimation; negative values indicate underestimation. Bias is generally positive on MIMIC and negative on CODE-15. Curves and bands show the mean bias and one SD across seeds.'
if age_bias_summary.n.eq(2).all():
    bias_caption += ' 2 seeds.'
else:
    for site, site_label in [('mimic', 'MIMIC'), ('code15', 'CODE-15')]:
        counts = age_bias_summary.loc[age_bias_summary.site.eq(site), 'n']
        if len(counts) and counts.eq(2).all():
            bias_caption += f' {site_label}: 2 seeds.'
bias_tex = '\n'.join(['\\begin{figure}[h]', '\\begin{center}', f'\\includegraphics[width={bias_width / PAPER_WIDTH_IN:.6f}\\textwidth]{{figures/fig14_age_late_fusion_bias.pdf}}', '\\end{center}', '\\caption{' + latex_escape(bias_caption) + '}', '\\label{fig:age-late-fusion-bias}', '\\end{figure}', ''])
(OUT / 'figures/fig14_age_late_fusion_bias.tex').write_text(bias_tex, encoding='utf-8')
print(bias_tex)
